# Predictive Maintenance — Prototype Notebook (Unit 2b)

This notebook is the test bed for Layer 3's Predictive Maintenance unit, before anything gets wired into the real pipeline.

**What this notebook proves, in order:**
1. A synthetic degrading machine vs. a healthy machine, using your real meter fields
2. Preprocessing: trend features computed the same way the real pipeline will compute them
3. Stage A — an unsupervised Health Index (works with zero failure history)
4. Early-warning change-point detection on top of the Health Index
5. Stage B — a supervised risk classifier, demonstrated on synthetic labels (swap in real logged maintenance events later)
6. The exact output payload this unit will emit into the Layer 3 schema
7. How this plugs into the real pipeline as a scheduled batch job

**Why two stages, not one supervised model straight away:** a real RUL/failure model needs run-to-failure history, which doesn't exist yet. Stage A works from day one off physics/statistics alone; Stage B activates once even a handful of real maintenance events are logged. Same cold-start philosophy as the anomaly detection unit.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
%matplotlib inline

## 2. Synthetic data — two machines

One healthy pump (flat signatures), one degrading pump (harmonics, PF, imbalance, and neutral current all drift over 60 days). Fields match your real `header.csv` schema: `Current_R/Y/B_Harm`, `PF_Ave`, `Neutral_current`, plus a derived `phase_imbalance_pct` (computed in the real pipeline by Unit 1, not raw from the meter).

Replace this cell with real historical data once you have it — the rest of the notebook is agnostic to where the dataframe comes from, as long as the column names match.

In [ ]:
def generate_machine_history(machine_id, days=60, degrading=False):
    dates = pd.date_range('2026-01-01', periods=days, freq='D')
    t = np.arange(days)
    # degradation ramps 0 -> 1 between day 20 and day 60; 0 for a healthy machine
    degradation = np.clip((t - 20) / 40, 0, 1) if degrading else np.zeros(days)
    df = pd.DataFrame({
        'date': dates,
        'machine_id': machine_id,
        'Current_R_Harm': 3.0 + degradation * 9.0 + np.random.normal(0, 0.3, days),
        'Current_Y_Harm': 3.0 + degradation * 8.5 + np.random.normal(0, 0.3, days),
        'Current_B_Harm': 3.0 + degradation * 9.2 + np.random.normal(0, 0.3, days),
        'PF_Ave': 0.95 - degradation * 0.30 + np.random.normal(0, 0.01, days),
        'Neutral_current': 1.0 + degradation * 4.0 + np.random.normal(0, 0.1, days),
        'phase_imbalance_pct': 2.0 + degradation * 10.0 + np.random.normal(0, 0.3, days),
    })
    return df

df = pd.concat([
    generate_machine_history('pump_01_healthy', degrading=False),
    generate_machine_history('pump_02_degrading', degrading=True),
]).reset_index(drop=True)

df.head()

## 3. Preprocessing — trend features

Predictive maintenance is a **slow signal**: a single reading means little, a rising or falling trend over days/weeks means a lot. This is the key difference from the anomaly detection unit, which reacts to individual readings.

`rolling_slope` fits a line over a trailing window and returns its slope — positive slope on harmonics/imbalance/neutral current means things are getting worse; negative slope on PF means efficiency is declining. In the real pipeline, this reuses the rolling-window infrastructure Unit 1 already computes — no separate feature pipeline needed.

In [ ]:
def rolling_slope(series, window):
    slopes = []
    for i in range(len(series)):
        if i < window - 1:
            slopes.append(np.nan)
            continue
        y = series.iloc[i - window + 1:i + 1].values
        x = np.arange(window)
        slopes.append(np.polyfit(x, y, 1)[0])
    return pd.Series(slopes, index=series.index)

def add_trend_features(g, window=7):
    g = g.sort_values('date').copy()
    for col in ['Current_R_Harm', 'Current_Y_Harm', 'Current_B_Harm',
                'PF_Ave', 'Neutral_current', 'phase_imbalance_pct']:
        g[f'{col}_slope_{window}d'] = rolling_slope(g[col], window)
    return g

# loop per machine explicitly (groupby().apply() can silently drop the group key column
# in some pandas versions when the function returns the full frame — this avoids that)
df_feat = pd.concat([add_trend_features(g) for _, g in df.groupby('machine_id')]).reset_index(drop=True)

df_feat.tail()

## 4. Stage A — unsupervised Health Index

No failure labels needed. Compares each machine's *current* readings against a population-prior baseline (mean/std for a healthy machine of this equipment class — in production this comes straight from Unit 3's Stage 1 priors, not hardcoded like here) and converts the average deviation into a single 0-100 health score.

In [ ]:
# Population-prior baseline for a healthy machine of this equipment class.
# In the real pipeline this is read from Layer 0 config / Unit 3's Stage 1 priors.
BASELINE = {
    'Current_R_Harm': (3.0, 0.5), 'Current_Y_Harm': (3.0, 0.5), 'Current_B_Harm': (3.0, 0.5),
    'PF_Ave': (0.95, 0.02), 'Neutral_current': (1.0, 0.2), 'phase_imbalance_pct': (2.0, 0.5),
}

def compute_health_index(row):
    z_scores = [abs(row[col] - mean) / std for col, (mean, std) in BASELINE.items()]
    avg_z = np.mean(z_scores)
    return max(0, 100 - avg_z * 12)

df_feat['health_index'] = df_feat.apply(compute_health_index, axis=1)

df_feat.groupby('machine_id')['health_index'].agg(['min', 'max'])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for mid, g in df_feat.groupby('machine_id'):
    ax.plot(g['date'], g['health_index'], label=mid)
ax.axhline(50, color='red', linestyle='--', label='Example risk threshold')
ax.set_ylabel('Health index (0-100)')
ax.set_title('Health index over time')
ax.legend()
plt.show()

## 5. Early-warning change-point detection (CUSUM)

The Health Index tells you *how bad it is now*. CUSUM tells you *when it started changing* — useful as an earlier trigger than waiting for the index to cross a fixed threshold. This mirrors the persistence-gate philosophy from the anomaly detection unit: don't react to one bad reading, react to a sustained shift.

In [ ]:
def cusum_flags(series, threshold=8, drift=0.5):
    s_pos, s_neg, flags = 0, 0, []
    baseline_mean = series.dropna().iloc[:10].mean()
    for val in series.fillna(method='ffill'):
        s_pos = max(0, s_pos + (baseline_mean - val) - drift)
        s_neg = min(0, s_neg + (baseline_mean - val) + drift)
        flags.append(s_pos > threshold or abs(s_neg) > threshold)
    return flags

for mid, g in df_feat.groupby('machine_id'):
    g = g.sort_values('date')
    flags = cusum_flags(g['health_index'])
    first_flag = next((i for i, f in enumerate(flags) if f), None)
    when = g['date'].iloc[first_flag] if first_flag is not None else 'never'
    print(f"{mid}: first change-point flagged at {when}")

## 6. Stage B — supervised risk classifier (demo only)

**This cell uses synthetic labels** (`health_index < 50`) purely to demonstrate the mechanics. In production, replace `will_need_maintenance_14d` with real logged maintenance/failure events once you have even 10-20 of them across your fleet — that's enough to start training on, it doesn't need to be a large dataset.

Random Forest is the right first choice here: works well on small tabular datasets, gives you feature importances for free (useful for explaining *why* a machine is flagged, which matters in a demo/presentation), and doesn't need much tuning.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

df_feat['will_need_maintenance_14d'] = (df_feat['health_index'] < 50).astype(int)  # placeholder for real labels

feature_cols = [c for c in df_feat.columns if c.endswith('_slope_7d')]
train = df_feat.dropna(subset=feature_cols)
X, y = train[feature_cols], train['will_need_maintenance_14d']

clf = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=42)
clf.fit(X, y)

pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=False)

## 7. Output — wrapped into the Layer 3 schema

This is what actually leaves the unit. Matches the `event_type: maintenance_risk` variant of the standard Layer 3 output contract (see `Layer3_Model_Layer_Technical_Architecture.md`, §3) — Layer 4/5 consume this exact shape regardless of which stage produced it.

In [ ]:
import uuid

def predict_maintenance_risk(machine_id, latest_row, confidence_stage='stage_a_unsupervised'):
    health_index = latest_row['health_index']
    risk_tier = 'high' if health_index < 40 else 'medium' if health_index < 65 else 'low'
    return {
        "event_id": str(uuid.uuid4()),
        "machine_id": machine_id,
        "timestamp": str(latest_row['date']),
        "source_unit": "predictive_maintenance",
        "event_type": "maintenance_risk",
        "severity": "alert" if risk_tier == "high" else "info",
        "health_index": round(float(health_index), 1),
        "risk_tier": risk_tier,
        "confidence_stage": confidence_stage,
        "contributing_features": {
            "current_r_harm_slope_7d": round(float(latest_row['Current_R_Harm_slope_7d']), 3),
            "pf_ave_slope_7d": round(float(latest_row['PF_Ave_slope_7d']), 4),
            "phase_imbalance_slope_7d": round(float(latest_row['phase_imbalance_pct_slope_7d']), 3),
        },
    }

latest = df_feat[df_feat['machine_id'] == 'pump_02_degrading'].dropna(subset=['health_index']).iloc[-1]
predict_maintenance_risk('pump_02_degrading', latest)

## 8. Plugging this into the real pipeline

**Cadence:** run as a scheduled batch job (daily is enough — this is a slow signal), not per-reading real-time like the anomaly detector.

**Inputs:** reuses Unit 1's already-computed rolling features directly. No separate ingestion path needed — same normalized, context-tagged stream the anomaly detector reads from.

**Integration contract:** the unit is one function —
```python
def predict_maintenance_risk(machine_id: str, latest_feature_row: dict, confidence_stage: str) -> dict:
    ...  # returns the schema payload above
```
Called once per machine per batch cycle, output pushed to Unit 5 (Model Output & Serving) exactly like every other engine — Layer 4/5 don't need to know this came from a Random Forest vs. a z-score formula.

**Stage rollout plan:**
| Stage | Trigger to activate | What changes |
|---|---|---|
| A (this notebook, §4) | Day 1, always on | Health index from population priors |
| B (this notebook, §6) | ~10-20 real logged maintenance events | Swap synthetic labels for real ones, retrain periodically |
| C (not in this notebook) | Enough run-to-failure histories across the fleet (months out) | Survival analysis (Cox / Weibull) for actual RUL-in-days, not just a risk tier |

**Before presenting:** be upfront that Stage B here runs on synthetic labels as a mechanics demo, not a validated model — same honesty principle used for the anomaly detection cold-start story. It holds up better under questioning than overclaiming.